In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def standardize_publication_date(df, date_col='publication_date'):
    """
    Cleans and converts publication dates to datetime, handling:
    - Timezone info in parentheses (e.g., (UTC+02))
    - Two-digit or four-digit years
    - Day-first formats (DD/MM/YY or DD/MM/YYYY)
    Adds a 'publication_year' column.
    """
    df = df.copy()
    
    # 1. Remove timezone info in parentheses
    df[date_col] = df[date_col].astype(str).str.replace(r"\s*\(UTC[+-]\d{2}\)", "", regex=True).str.strip()
    
    # 2. Attempt parsing with day-first=True
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    
    # 3. Warn if parsing failed
    if df[date_col].isna().any():
        print(f"⚠️ Warning: {df[date_col].isna().sum()} dates could not be parsed in {date_col}")
    
    # 4. Create publication_year column
    df['publication_year'] = df[date_col].dt.year
    
    return df


1. Read exported ted files from expert query. Clean, standardize

In [ ]:
file_path = r"ted_download\TED_06-10-2025_FINAL.csv"

# Read into a DataFrame
df = pd.read_csv(file_path, encoding="utf-8", low_memory=False, )

# Rename specific columns first
df = df.rename(columns={
    "Official name" :  "buyer_name",
    "Country.1": "winner_country",
    "Country": "buyer_country",
    "Town": "buyer_town",
    "Internet address": "buyer_internet_address",
    "Organisation filling this role - Town": "winner_town",
    "Title": "contract_title",
    "Official name.1": "winner_name"
})

# Then clean up all column names (spaces, dots, etc.) and make them lowercase
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\.]+", "_", regex=True)
)



# Identify rows where 'unnamed_23' is not NaN (likely misaligned)
mask_shifted = df["unnamed:_23"].notna()

# Shift 'contract_title' and 'unnamed_23' one column to the left
df.loc[mask_shifted, "winner_town"] = df.loc[mask_shifted, "contract_title"]
df.loc[mask_shifted, "contract_title"] = df.loc[mask_shifted, "unnamed:_23"]

# Drop the now-empty 'unnamed_23' column
df = df.drop(columns=["unnamed:_23", "approximate_value_of_the_framework_agreements"])

In [ ]:
df = standardize_publication_date(df, 'publication_date')

In [ ]:
df

In [ ]:
# Filter out rows where winner_selection_status indicates no winner
exclude_phrases = ["The winner was not yet chosen", "No winner was chosen"]

def filter_no_winner(df, status_col='winner_selection_status'):
    """
    Removes rows where winner_selection_status indicates no winner.
    """
    mask = df[status_col].str.startswith(tuple(exclude_phrases), na=False)
    return df[~mask]

# Apply filtering
df = filter_no_winner(df)

In [ ]:
df

In [ ]:
# Columns that may have multiple comma-separated entries
cols_to_split = ['winner_name']

# Replace any potential missing column errors (keep only existing ones)
cols_to_split = [c for c in cols_to_split if c in df.columns]

for col in cols_to_split:
    # Split only non-null cells, leave NaNs as-is
    df[col] = df[col].apply(lambda x: str(x).split(',') if pd.notna(x) else np.nan)
    df = df.explode(col).reset_index(drop=True)
    # Strip whitespace safely, ignoring NaN
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)


In [ ]:
df

In [ ]:
df.info()

2. Read downloaded ted csv files (CAN_2023_2018). Clean, standardize

In [ ]:
file_path = r"ted_download/export_CAN_2023_2018_relevant_cpvs.csv"

# Read CSV
df2 = pd.read_csv(file_path, encoding="utf-8", low_memory=False, index_col=0)

df2.head()

In [ ]:
df2.info()

In [ ]:
# Mapping from df2 columns → df column names
column_mapping = {
    "ID_NOTICE_CAN": "notice_publication_number",
    "CPV": "main_classification",
    "TYPE_OF_CONTRACT": "main_nature_of_the_contract",
    "ADDITIONAL_CPVS": "additional_classification",
    "TAL_LOCATION_NUTS": "place_of_performance",
    "DT_DISPATCH": "publication_date",
    "MAIN_ACTIVITY": "activity_of_the_contracting_authority",
    "ISO_COUNTRY_CODE": "buyer_country",
    "CAE_NAME": "official_name",
    "CAE_NATIONALID": "registration_number",
    "CAE_TYPE": "legal_type_of_the_buyer",
    "B_ON_BEHALF": "legal_basis_of_the_procedure",
    "WIN_COUNTRY_CODE": "winner_country",
    "CAE_NAME": "buyer_name",
    "WIN_NAME": "winner_name",
    "B_FRA_AGREEMENT": "social_objective_promoted",
    "WIN_TOWN": "winner_town",
    "TITLE": "contract_title",
    "YEAR": "publication_year",
    "LOTS_NUMBER": "num_lots"
}

# Select only the relevant df2 columns that exist in df
relevant_df2_cols = [col for col in column_mapping if col in df2.columns]

# Create a new DataFrame with only relevant columns and renamed
df2_relevant = df2.rename(columns={k: column_mapping[k] for k in relevant_df2_cols})

# Define CPV codes of interest
cpv_codes = [18000000, 18100000, 18200000, 18300000, 18400000, 35200000, 39518000]

df2_relevant["buyer_country"] = df2_relevant["buyer_country"].str.strip().str.upper()


cpv_codes_numeric = [18000000, 18100000, 18200000, 18300000, 18400000, 35200000, 39518000]
df2_relevant["main_classification_numeric"] = pd.to_numeric(df2_relevant["main_classification"], errors="coerce")

# 3. Filter rows
df2_relevant = df2_relevant[
    (df2_relevant["buyer_country"] == "DE") &
    (df2_relevant["main_classification_numeric"].isin(cpv_codes_numeric))]

df2_relevant

In [ ]:
df2_relevant.info()

In [ ]:
df2_relevant = standardize_publication_date(df2_relevant, 'publication_date')

# 2. Count unique winner names
df2_relevant["notice_publication_number"].nunique()

In [ ]:
df2_relevant

In [ ]:
df2_relevant.info()

In [ ]:
def missing_winner_summary(df, winner_col='winner_name', year_col='publication_year'):
    """
    Calculates missing winner names per year and their proportion.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing winner names and publication year.
    winner_col : str
        Column name for winner names.
    year_col : str
        Column name for publication year.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns: publication_year, missing_count, total_count, missing_percentage
    """
    # Count missing winner names per year
    missing_winners_by_year = (
        df[df[winner_col].isna() | (df[winner_col].astype(str).str.strip() == '')]
        .groupby(year_col)
        .size()
        .reset_index(name="missing_count")
    )

    # Total rows per year
    total_by_year = df.groupby(year_col).size().reset_index(name="total_count")

    # Merge and calculate percentage
    missing_summary = missing_winners_by_year.merge(total_by_year, on=year_col, how="right")
    missing_summary["missing_count"] = missing_summary["missing_count"].fillna(0).astype(int)
    missing_summary["missing_percentage"] = (
        missing_summary["missing_count"] / missing_summary["total_count"] * 100
    ).round(2)

    return missing_summary.sort_values(year_col)


In [ ]:
missing_winner_summary(df, winner_col='winner_name', year_col='publication_year')


In [ ]:
missing_winner_summary(df2_relevant, winner_col='winner_name', year_col='publication_year')

In [ ]:
def count_column_matches(df1, df2, column="buyer_name"):
    """
    Count how many values in `column` of df1 appear in `column` of df2.

    Parameters:
        df1 (pd.DataFrame): First DataFrame.
        df2 (pd.DataFrame): Second DataFrame.
        column (str): Column name to match on (default 'buyer_name').

    Returns:
        int: Number of matches.
    """
    # Ensure both columns exist
    if column not in df1 or column not in df2:
        raise ValueError(f"Column '{column}' must exist in both DataFrames.")

    # Convert to string
    col1 = df1[column].astype(str)
    col2 = df2[column].astype(str)

    # Count matches
    return col1.isin(col2).sum()



In [ ]:
count_column_matches(df, df2_relevant, "notice_publication_number")

In [ ]:
def find_missing_winners(df_main, df_reference, key="notice_publication_number", winner_col="winner_name"):
    """
    Find rows in df_main where the winner_name is missing but exists in df_reference.

    Parameters:
        df_main (pd.DataFrame): Main DataFrame to check.
        df_reference (pd.DataFrame): Reference DataFrame containing winner info.
        key (str): Column to join on (default 'notice_publication_number').
        winner_col (str): Name of winner column (default 'winner_name').

    Returns:
        pd.DataFrame: Rows from df_main with missing winner_name but available in df_reference.
    """
    # Ensure join keys are strings and stripped
    df_main[key] = df_main[key].astype(str).str.strip()
    df_reference[key] = df_reference[key].astype(str).str.strip()

    # Prepare reference winners
    df_ref_winners = df_reference[[key, winner_col]].dropna(subset=[winner_col])

    # Merge temporarily
    merged = df_main.merge(
        df_ref_winners,
        on=key,
        how="left",
        suffixes=("", "_from_ref")
    )

    # Filter where main winner is missing but reference has it
    missing_available = merged[
        merged[winner_col].isna() & merged[f"{winner_col}_from_ref"].notna()
    ]

    # Return key + reference winner, dropping duplicates
    return missing_available[[key, f"{winner_col}_from_ref"]].drop_duplicates()


In [ ]:
missing_winners = find_missing_winners(df2_relevant, df3)
print(f"🔍 {len(missing_winners)} rows where df winner_name is missing but df3 provides one")
missing_winners

In [ ]:
import pandas as pd

def fill_and_append_multiple(df_main, df_references, key="notice_publication_number", winner_col="winner_name"):
    """
    Fill missing winner_name in df_main using one or multiple reference DataFrames.
    Append new rows from references if key does not exist in df_main.
    """
    # Accept single DataFrame input as well
    if isinstance(df_references, pd.DataFrame):
        df_references = [df_references]

    df_main[key] = df_main[key].astype(str).str.strip()
    updated_main = df_main.copy()

    for df_ref in df_references:
        df_ref[key] = df_ref[key].astype(str).str.strip()
        df_ref_winners = df_ref.dropna(subset=[winner_col])

        merged = updated_main.merge(
            df_ref_winners,
            on=key,
            how="left",
            suffixes=("", "_from_ref")
        )
        merged[winner_col] = merged[winner_col].combine_first(merged[f"{winner_col}_from_ref"])
        merged = merged.drop(columns=[f"{winner_col}_from_ref"])

        updated_main = merged

        # Append new rows from reference not in main
        new_rows = df_ref_winners[~df_ref_winners[key].isin(updated_main[key])]
        updated_main = pd.concat([updated_main, new_rows])

    return updated_main


In [ ]:
df2_filled = fill_and_append_multiple(df2_relevant, df)
df2_filled

In [ ]:
df2_filled = fill_and_append_multiple(df2_relevant, df, "buyer_town")
df2_filled

In [ ]:
df2_filled = df2_filled.dropna(axis=1, how="all")
df2_filled

In [ ]:
df2_filled = df2_filled.drop_duplicates()

In [ ]:
df2_filled

In [ ]:
#df2_filled.to_csv("curated_data_2018-2025.csv")

In [ ]:
count_column_matches(df2_filled, df3, "notice_publication_number")

In [ ]:
count_column_matches(df2_filled, df, "notice_publication_number")

In [ ]:
df2_filled.winner_name.isna().sum()

In [ ]:
missing_winner_summary(df2_filled)